# 입찰메이트 RAG — 서빙 E2E 평가 (kh_v3, A100 정리본)

KURE + Phi-4-mini(LoRA FT) 단일 시나리오. 진단/패치 셀을 본문에 흡수하고 A100 기준으로 정리.

**실행 순서**: `[0]` 설치 → `[0b]` 진단(컬렉션·메타키 확인) → `[1]` 데이터 준비 → `[2]` config → `[3]` pre-check → `[4]` retriever → `[5]` generator → `[6]` 스모크 → `[7]` 579행 생성 → `[8]` 무결성 → `[9]` Retrieval 지표 → `[10]` Judge → `[11]` Gen 요약 → `[12]` Release Gate → `[13]` 산출물 → `[14]` 정성분석

> **kh_v3 설정값**은 `[1]` 셀 상단 한 곳에 모아둠. 파일명/컬렉션명이 확실하지 않으면 `[0b]`를 먼저 돌려 확인 후 채울 것.

In [ ]:
# [0] 설치
!pip install -q chromadb sentence-transformers rank_bm25 kiwipiepy peft transformers accelerate openai tqdm nest_asyncio rapidfuzz
!pip uninstall -y torchao
print("설치 완료 — 런타임 재시작 메시지 뜨면 재시작 후 [0b]부터")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 24.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.4 MB/s eta 0:00:00
   ━━━━

In [ ]:
# [0a] 드라이브 마운트 + bidmate 폴더 확인
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/data/bidmate'

# kh_v3 핵심 파일 존재/크기 확인 (업로드 누락 조기 감지)
for need in ['kh_v3_chroma_FULL.tar.gz',
             'bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl',
             'chunks/kh_v3.json',
             'eval/eval_retrieval_579.csv']:
    fp = f'{DRIVE}/{need}'
    if os.path.exists(fp):
        print(f'OK       {os.path.getsize(fp)/1e6:7.0f}MB  {need}')
    else:
        print(f'MISSING            {need}')

# 특히 chroma tar 는 1700MB 근처여야 정상 (업로드 중단 시 작게 나옴)
_tar = f'{DRIVE}/kh_v3_chroma_FULL.tar.gz'
if os.path.exists(_tar):
    _sz = os.path.getsize(_tar)/1e6
    print(f'\nchroma tar: {_sz:.0f}MB', '✅ 정상' if _sz > 1000 else '⚠️ 너무 작음 — 업로드 미완 의심')


Mounted at /content/drive
OK          1787MB  kh_v3_chroma_FULL.tar.gz
OK            73MB  bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl
OK           185MB  chunks/kh_v3.json
OK             1MB  eval/eval_retrieval_579.csv

chroma tar: 1787MB ✅ 정상


In [ ]:
# [0b] 진단 — tar/로컬 chroma 안의 컬렉션 이름·개수 + 메타 기관키 자동 확인 (마운트 후 1회)
#   여기서 나온 count>0 인 이름을 [1] config 의 COLLECTION_NAME 에 그대로 넣으세요.
#   '감지된 기관키' 도 [1] 의 AGENCY_KEY 에 반영(기본 auto 면 안 넣어도 됨).
import os, shutil, tarfile, gc, chromadb
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass
DRIVE = '/content/drive/MyDrive/data/bidmate'

_AGENCY_CANDIDATES = ['agency', 'organization_cleaned', 'organization', 'agency_name', 'institution']

def _detect_agency_key(metas):
    for k in _AGENCY_CANDIDATES:
        if any(isinstance(m, dict) and m.get(k) for m in metas):
            return k
    # 후보에 없으면 값이 채워진 첫 문자열 키
    for m in metas:
        if isinstance(m, dict):
            for k, v in m.items():
                if isinstance(v, str) and v.strip():
                    return k
    return None

def list_collections(chroma_dir):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    client = chromadb.PersistentClient(path=chroma_dir)
    cols = client.list_collections()
    print(f'  경로: {chroma_dir}')
    if not cols: print('  (컬렉션 없음)')
    for c in cols:
        try:
            col = client.get_collection(c.name)
            cnt = col.count()
            sample = col.get(limit=20, include=['metadatas'])['metadatas'] or []
            key = _detect_agency_key(sample)
            keys = list(sample[0].keys()) if sample else []
            print(f'  - {c.name:34} count={cnt:>8,}  | 감지된 기관키={key}  | 메타키={keys[:8]}')
        except Exception as e:
            print(f'  - {c.name:34} (count/메타 실패: {e})')

# 1) 로컬에 이미 풀려있으면 표시 (count 0 이면 그 폴더는 쓰면 안 됨)
for p in ['/content/bidmate_kh_v3/chroma_db', '/content/chroma_db']:
    if os.path.isdir(p):
        print('▶ 로컬 chroma'); list_collections(p)

# 2) 드라이브 tar.gz 를 임시로 풀어 확인 (정상 데이터의 출처)
tar = f'{DRIVE}/chroma_db.tar.gz'
tmp = '/content/_chroma_probe'
if os.path.exists(tar):
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    shutil.copy(tar, f'{tmp}/c.tar.gz')
    with tarfile.open(f'{tmp}/c.tar.gz') as t: t.extractall(tmp, filter='data')
    src = next((root for root,_,fs in os.walk(tmp) if 'chroma.sqlite3' in fs), None)
    print('▶ tar 내 sqlite:', src)
    if src: list_collections(src)
elif os.path.isdir(f'{DRIVE}/chroma_db'):
    print('▶ 드라이브 폴더 chroma'); list_collections(f'{DRIVE}/chroma_db')
print('\n※ count>0 인 이름 → [1] COLLECTION_NAME / 감지된 기관키 → [1] AGENCY_KEY(기본 auto)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
▶ tar 내 sqlite: /content/_chroma_probe/chroma_db
  경로: /content/_chroma_probe/chroma_db
  - bidmate_chunks_all                 count=  10,068  | 감지된 기관키=agency  | 메타키=['year', 'agency', 'domain', 'has_table', 'project', 'char_len', 'meta_h1', 'source_file']
  - bidmate_retrieval_v1               count=  26,316  | 감지된 기관키=agency  | 메타키=['meta_h2', 'parent_id', 'has_table', 'meta_h1', 'agency', 'parent_text', 'char_len', 'domain']

※ count>0 인 이름 → [1] COLLECTION_NAME / 감지된 기관키 → [1] AGENCY_KEY(기본 auto)


In [ ]:
# [1] 마운트 + chroma 로컬 준비  (★ 빈 컬렉션 재사용 방지 — 0점 버그 핵심)
#  ┌──────────────────────────────────────────────────────────────┐
#  │  kh_v3 설정 — 이 블록만 맞으면 나머지 셀은 전부 자동으로 따라감.   │
#  │  파일명/컬렉션명이 불확실하면 [0b] 진단 결과로 채울 것.            │
#  └──────────────────────────────────────────────────────────────┘

import os, shutil, tarfile, gc, chromadb
DRIVE = '/content/drive/MyDrive/data/bidmate'

# ===== kh_v3 선택 =================================================
CHUNK_TAG       = 'kh_v3'
CHUNK_FILE      = 'chunks/kh_v3.json'
BM25_FILE       = 'bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl'
#CHROMA_TAR      = 'kh_v3_chroma_FULL.tar.gz'
CHROMA_TAR = 'kh_v3_chroma_FIXED.tar.gz'   # ← 복구본
CHROMA_SUBDIR   = 'chroma_db_kh_v3_clean'
COLLECTION_NAME = 'bidmate_kh_v3_KURE_PHI'
EXPECT_MIN      = 35000
AGENCY_KEY      = 'auto'
SIG_TH          = 0.5   # D타입 거절 임계값(reranker sigmoid). 기존과 동일
# =================================================================

LOCAL      = f'/content/bidmate_{CHUNK_TAG}'
CHROMA_DIR = f'{LOCAL}/{CHROMA_SUBDIR}'
os.makedirs(LOCAL, exist_ok=True)
COLLECTION_NAME = COLLECTION_NAME.strip()

def _clear():
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass

def _count(path, name):
    _clear()
    try: return chromadb.PersistentClient(path=path).get_collection(name).count()
    except Exception: return -1

def _ensure_chroma():
    # count 가 EXPECT_MIN 이상일 때만 재사용. 0/미달이면 무조건 tar 재해제.
    n = _count(CHROMA_DIR, COLLECTION_NAME)
    if n >= EXPECT_MIN:
        print(f'chroma 재사용: {CHROMA_DIR} (count={n:,})'); return
    print(f'재적재 필요 (현재 count={n}) — 기존 폴더 비우고 tar 재해제')
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)
    tar = f'{DRIVE}/{CHROMA_TAR}'
    assert os.path.exists(tar), f'tar 없음: {tar}'
    sz = os.path.getsize(tar)/1e6
    assert sz > 1, f'❌ tar 이 너무 작음({sz:.2f}MB) — 빈 파일 의심: {tar}'
    print(f'tar 해제 중: {CHROMA_TAR} ({sz:.0f}MB)')
    with tarfile.open(tar) as t:
        assert any('chroma.sqlite3' in nm for nm in t.getnames()), '❌ tar 안에 sqlite 없음'
        t.extractall(LOCAL, filter='data')
    src = next((root for root,_,fs in os.walk(LOCAL) if 'chroma.sqlite3' in fs), None)
    assert src, '해제 후 sqlite 못 찾음'
    if os.path.abspath(src) != os.path.abspath(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR, ignore_errors=True); shutil.move(src, CHROMA_DIR)
    print('chroma 준비 완료:', CHROMA_DIR)
    n2 = _count(CHROMA_DIR, COLLECTION_NAME)
    assert n2 >= EXPECT_MIN, (
        f'❌ tar 해제 후에도 {COLLECTION_NAME} count={n2} (< {EXPECT_MIN}). '
        f'[0b] 에서 tar 안 컬렉션 이름을 다시 확인하세요.')
    print(f'재검증 통과: {COLLECTION_NAME} count={n2:,}')

_ensure_chroma()

_cl = chromadb.PersistentClient(path=CHROMA_DIR)
_names = [c.name for c in _cl.list_collections()]
assert COLLECTION_NAME in _names, f'컬렉션 {COLLECTION_NAME} 없음. 존재: {_names}'
_col = _cl.get_collection(COLLECTION_NAME)
EXPECT_N = _col.count()
assert EXPECT_N >= EXPECT_MIN, f'❌ EXPECT_N={EXPECT_N} 비정상 — 재실행 필요'

# ── 기관키 자동 감지 ───────────────────────────────────────────
_AGENCY_CANDIDATES = ['agency','organization_cleaned','organization','agency_name','institution']
if AGENCY_KEY == 'auto':
    _metas = _col.get(limit=30, include=['metadatas'])['metadatas'] or []
    AGENCY_KEY = next((k for k in _AGENCY_CANDIDATES
                       if any(isinstance(m,dict) and m.get(k) for m in _metas)), None)
    if AGENCY_KEY is None:
        for m in _metas:
            if isinstance(m, dict):
                AGENCY_KEY = next((k for k,v in m.items() if isinstance(v,str) and v.strip()), None)
                if AGENCY_KEY: break
    assert AGENCY_KEY, '❌ 기관키 자동 감지 실패 — [0b] 로 메타키 확인 후 AGENCY_KEY 직접 지정'
print(f'[{CHUNK_TAG}] 컬렉션={COLLECTION_NAME} | count={EXPECT_N:,} | 기관키={AGENCY_KEY!r} | 그 외={_names}')

# 청크/bm25/eval 로컬 복사
for rel in [CHUNK_FILE, BM25_FILE, 'eval/eval_retrieval_579.csv']:
    s=f'{DRIVE}/{rel}'; d=f'{LOCAL}/{rel}'
    assert os.path.exists(s), f'원본 없음: {s}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d): shutil.copy(s, d)

# 후속 셀 고정 경로 동기화 + 출력 폴더 청킹별 분리
FIXED='/content/bidmate'
os.makedirs(f'{FIXED}/eval', exist_ok=True)
shutil.copy(f'{LOCAL}/eval/eval_retrieval_579.csv', f'{FIXED}/eval/eval_retrieval_579.csv')
OUT_TAG_DIR = f'{FIXED}/outputs/{CHUNK_TAG}'
os.makedirs(OUT_TAG_DIR, exist_ok=True)
print('eval 동기화 / 출력폴더:', OUT_TAG_DIR)

_CHUNK_TAG,_CHUNK_FILE,_BM25_FILE,_COLLECTION_NAME,_EXPECT_N,_CHROMA_DIR,_OUT_TAG_DIR,_AGENCY_KEY,_SIG_TH = \
    CHUNK_TAG,CHUNK_FILE,BM25_FILE,COLLECTION_NAME,EXPECT_N,CHROMA_DIR,OUT_TAG_DIR,AGENCY_KEY,SIG_TH

재적재 필요 (현재 count=-1) — 기존 폴더 비우고 tar 재해제
tar 해제 중: kh_v3_chroma_FULL.tar.gz (1787MB)
chroma 준비 완료: /content/bidmate_kh_v3/chroma_db_kh_v3_clean
재검증 통과: bidmate_kh_v3_KURE_PHI count=38,287
[kh_v3] 컬렉션=bidmate_kh_v3_KURE_PHI | count=38,287 | 기관키='agency' | 그 외=['bidmate_chunks_all', 'bidmate_chunks_all_B', 'bidmate_retrieval_v1', 'bidmate_kh_fixed_v1_B', 'bidmate_retrieval_openai_v1', 'bidmate_kh_v3_A-1', 'bidmate_kh_v3_B', 'bidmate_kh_fixed_v1_A-2', 'bidmate_kh_fixed_v2_B', 'bidmate_kh_fixed_v1_A-1', 'bidmate_chunks_all_A-1', 'bidmate_kure', 'bidmate_kh_v3_KURE_PHI', 'bidmate_kh_v3_A-2', 'bidmate_kh_fixed_v2_A-2', 'bidmate_retrieval_koe5_v1', 'bidmate_kh_fixed_v2_A-1', 'bidmate_chunks_all_A-2']
eval 동기화 / 출력폴더: /content/bidmate/outputs/kh_v3


In [ ]:
import os, config as C
print('CHROMA_DIR:', C.CHROMA_PATH)
for f in sorted(os.listdir(C.CHROMA_PATH)):
    full = os.path.join(str(C.CHROMA_PATH), f)
    if os.path.isdir(full):
        print(f'  [DIR] {f}/  → {os.listdir(full)[:6]}')
    else:
        print(f'  {f}  ({os.path.getsize(full)/1e6:.1f}MB)')

CHROMA_DIR: /content/bidmate_kh_v3/chroma_db_kh_v3_clean
  ._chroma.sqlite3  (0.0MB)
  ._data_level0.bin  (0.0MB)
  ._header.bin  (0.0MB)
  ._index_metadata.pickle  (0.0MB)
  ._length.bin  (0.0MB)
  ._link_lists.bin  (0.0MB)
  [DIR] 21be2de2-405f-46c0-b122-611962dac9bb/  → ['length.bin', 'link_lists.bin', 'data_level0.bin', 'header.bin']
  chroma.sqlite3  (5000.2MB)
  data_level0.bin  (160.5MB)
  header.bin  (0.0MB)
  index_metadata.pickle  (2.1MB)
  length.bin  (0.2MB)
  link_lists.bin  (0.3MB)


In [ ]:
import sqlite3, config as C
con = sqlite3.connect(f'{C.CHROMA_PATH}/chroma.sqlite3')
rows = con.execute("""
  SELECT s.id, c.name FROM segments s
  JOIN collections c ON s.collection = c.id
  WHERE c.name = ?
""", (C.COLLECTION_NAME,)).fetchall()
print('세그먼트:', rows)
con.close()

세그먼트: [('21be2de2-405f-46c0-b122-611962dac9bb', 'bidmate_kh_v3_KURE_PHI'), ('ae19f335-b283-4b53-b3f1-5b05e2df028f', 'bidmate_kh_v3_KURE_PHI')]


In [ ]:
import os, config as C
root = str(C.CHROMA_PATH)

print('=== 루트 bin (정상 위치 아님) ===')
for f in ['data_level0.bin','header.bin','length.bin','link_lists.bin','index_metadata.pickle']:
    p = os.path.join(root, f)
    print(f'  {f:24} {os.path.getsize(p)/1e6:8.2f}MB' if os.path.exists(p) else f'  {f} 없음')

for uuid in ['21be2de2-405f-46c0-b122-611962dac9bb', 'ae19f335-b283-4b53-b3f1-5b05e2df028f']:
    d = os.path.join(root, uuid)
    print(f'\n=== UUID {uuid[:8]} 폴더 ===')
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            if f.startswith('._'): continue
            print(f'  {f:24} {os.path.getsize(os.path.join(d,f))/1e6:8.2f}MB')
    else:
        print('  (폴더 없음)')

=== 루트 bin (정상 위치 아님) ===
  data_level0.bin            160.49MB
  header.bin                   0.00MB
  length.bin                   0.15MB
  link_lists.bin               0.33MB
  index_metadata.pickle        2.12MB

=== UUID 21be2de2 폴더 ===
  data_level0.bin              0.42MB
  header.bin                   0.00MB
  length.bin                   0.00MB
  link_lists.bin               0.00MB

=== UUID ae19f335 폴더 ===
  (폴더 없음)


In [ ]:
# chroma bin 위치 복구 — 루트의 벡터 bin → 21be2de2 세그먼트 폴더로 이동
import os, shutil, gc, chromadb, config as C
root = str(C.CHROMA_PATH)
uuid = '21be2de2-405f-46c0-b122-611962dac9bb'
udir = os.path.join(root, uuid)

# chroma 클라이언트 핸들 정리 (파일 잠금 해제)
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass

# 0) ._ macOS 잔재 제거
removed = 0
for dirpath, _, files in os.walk(root):
    for f in files:
        if f.startswith('._'):
            os.remove(os.path.join(dirpath, f)); removed += 1
print(f'._ 잔재 제거: {removed}개')

# 1) 현재 21be2de2 폴더의 껍데기 bin 백업
bak = os.path.join(root, f'{uuid}__shell_backup')
os.makedirs(bak, exist_ok=True)
for f in ['data_level0.bin','header.bin','length.bin','link_lists.bin','index_metadata.pickle']:
    src_f = os.path.join(udir, f)
    if os.path.exists(src_f):
        shutil.move(src_f, os.path.join(bak, f))
print('기존 껍데기 백업 →', bak)

# 2) 루트의 진짜 bin → 21be2de2 폴더로 이동
moved = []
for f in ['data_level0.bin','header.bin','length.bin','link_lists.bin','index_metadata.pickle']:
    src_f = os.path.join(root, f)
    if os.path.exists(src_f):
        shutil.move(src_f, os.path.join(udir, f))
        moved.append(f)
print('루트 → 세그먼트 이동:', moved)

# 3) 이동 후 폴더 상태 확인
print('\n복구된 21be2de2 폴더:')
for f in sorted(os.listdir(udir)):
    print(f'  {f:24} {os.path.getsize(os.path.join(udir,f))/1e6:8.2f}MB')

._ 잔재 제거: 6개
기존 껍데기 백업 → /content/bidmate_kh_v3/chroma_db_kh_v3_clean/21be2de2-405f-46c0-b122-611962dac9bb__shell_backup
루트 → 세그먼트 이동: ['data_level0.bin', 'header.bin', 'length.bin', 'link_lists.bin', 'index_metadata.pickle']

복구된 21be2de2 폴더:
  data_level0.bin            160.49MB
  header.bin                   0.00MB
  index_metadata.pickle        2.12MB
  length.bin                   0.15MB
  link_lists.bin               0.33MB


In [ ]:
import gc, chromadb, config as C
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass

col = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME)
print('count:', col.count())
# 임베딩으로 직접 쿼리 테스트
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer(C.EMBED_MODEL_ID, device='cuda', cache_folder='/content/hf_cache/hub')
q = emb.encode(['한국가스공사 사업 예산'], normalize_embeddings=True)
res = col.query(query_embeddings=q.tolist(), n_results=5)
print('쿼리 결과 ids:', len(res['ids'][0]))
print('거리 샘플:', res['distances'][0][:3])

count: 38287


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

쿼리 결과 ids: 5
거리 샘플: [0.3917578458786011, 0.4495452642440796, 0.45616239309310913]


In [ ]:
# [2] config 주입
import json
import sys, types, os
from pathlib import Path
CODE='/content/drive/MyDrive/data/bidmate/code'
if CODE not in sys.path: sys.path.insert(0, CODE)
os.environ['HF_HOME']='/content/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

cfg=types.ModuleType('config')
cfg.ENV='colab'
cfg.PROJECT_ROOT=Path(f'/content/bidmate_{_CHUNK_TAG}')
cfg.DATASET_DIR=cfg.PROJECT_ROOT
cfg.CHUNKS_PATH=cfg.PROJECT_ROOT/_CHUNK_FILE
cfg.CHROMA_PATH=Path(_CHROMA_DIR)
cfg.BM25_PATH=cfg.PROJECT_ROOT/_BM25_FILE
cfg.EVAL_PATH=cfg.PROJECT_ROOT/'eval'
cfg.RESULT_DIR=cfg.PROJECT_ROOT/'eval_results'
cfg.ADAPTER_PATH=Path('/content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter')
cfg.LOG_PATH=cfg.PROJECT_ROOT/'web_user_access.log'
cfg.BASE_MODEL_ID='microsoft/Phi-4-mini-instruct'
cfg.LLM_MODEL='microsoft/Phi-4-mini-instruct'
cfg.EMBED_MODEL_ID='nlpai-lab/KURE-v1'
cfg.RERANKER_ID='BAAI/bge-reranker-v2-m3'
cfg.MAX_TOKENS_REWRITE=300; cfg.MAX_TOKENS_GENERATE=800
cfg.COLLECTION_NAME=_COLLECTION_NAME
cfg.EXPECT_N=_EXPECT_N
cfg.OUT_TAG_DIR=_OUT_TAG_DIR
cfg.AGENCY_KEY=_AGENCY_KEY
cfg.SIG_TH=_SIG_TH
cfg.DENSE_K=15; cfg.SPARSE_K=15; cfg.RRF_K=60; cfg.TOP_K=5
cfg.MMR_LAMBDA=0.6; cfg.MMR_TOP_N=20; cfg.RERANK_TOP_N=15; cfg.BATCH_SIZE=64
sys.modules['config']=cfg
assert cfg.EXPECT_N > 0, '❌ EXPECT_N=0 — [1] 재실행'
print(f'config OK → tag={_CHUNK_TAG} | col={cfg.COLLECTION_NAME} | expect={cfg.EXPECT_N:,} | agency_key={cfg.AGENCY_KEY!r} | sig_th={cfg.SIG_TH}')
print('  CHROMA:', cfg.CHROMA_PATH); print('  BM25  :', cfg.BM25_PATH); print('  OUT   :', cfg.OUT_TAG_DIR)

config OK → tag=kh_v3 | col=bidmate_kh_v3_KURE_PHI | expect=38,287 | agency_key='agency' | sig_th=0.5
  CHROMA: /content/bidmate_kh_v3/chroma_db_kh_v3_clean
  BM25  : /content/bidmate_kh_v3/bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl
  OUT   : /content/bidmate/outputs/kh_v3


In [ ]:
# [3] pre-check — GPU/A100 확인 + 파일 존재 + chroma count>0 최우선 검증
import torch, pickle, json, os, gc, chromadb
from pathlib import Path
import config as C

print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '❌ GPU 없음 — 런타임 유형을 A100 으로 변경'
gpu = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory/1e9,1)
print(f'  GPU : {gpu}  | VRAM: {vram} GB')
if 'A100' not in gpu:
    print(f'  ⚠️  A100 이 아님({gpu}) — 진행은 되지만 BATCH/속도 가정이 다를 수 있음')
DEVICE = 'cuda'

for k,p in {'CHUNKS':C.CHUNKS_PATH,'CHROMA':C.CHROMA_PATH,'BM25':C.BM25_PATH,
            'EVAL':C.EVAL_PATH/'eval_retrieval_579.csv','ADAPTER':C.ADAPTER_PATH}.items():
    print(f'{"OK" if Path(p).exists() else "MISSING":8}{k:8}{p}')

with open(C.CHUNKS_PATH, encoding='utf-8') as f:
    n_chunks = len(json.load(f))
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
n_chroma = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME).count()
with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
n_bm25 = len(bm['chunk_ids'])
print(f'\n청크JSON {n_chunks:,} | chroma {n_chroma:,} | bm25 {n_bm25:,}  (기준 chroma={C.EXPECT_N:,})')
if Path(C.ADAPTER_PATH).exists():
    print('어댑터:', os.listdir(C.ADAPTER_PATH)[:6])

assert n_chroma > 0, '❌ chroma 비어있음(count=0) — [1] 재실행해 tar 재해제'
assert n_chroma == C.EXPECT_N, f'chroma count 불일치: {n_chroma:,} != {C.EXPECT_N:,}'
if n_bm25 != n_chroma:
    print(f'⚠️  bm25({n_bm25:,}) != chroma({n_chroma:,}) — 하이브리드 인덱스 정합 확인')
print('✅ pre-check 통과')

CUDA: True
  GPU : NVIDIA A100-SXM4-40GB  | VRAM: 42.4 GB
OK      CHUNKS  /content/bidmate_kh_v3/chunks/kh_v3.json
OK      CHROMA  /content/bidmate_kh_v3/chroma_db_kh_v3_clean
OK      BM25    /content/bidmate_kh_v3/bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl
OK      EVAL    /content/bidmate_kh_v3/eval/eval_retrieval_579.csv
OK      ADAPTER /content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter

청크JSON 38,287 | chroma 38,287 | bm25 38,287  (기준 chroma=38,287)
어댑터: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
✅ pre-check 통과


In [ ]:
# import os
# import config as _C

# bad_file = f'{_C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv'
# if os.path.exists(bad_file):
#     os.remove(bad_file)
#     print(f"삭제 완료: {bad_file}")

삭제 완료: /content/bidmate/outputs/kh_v3/e2e_kure_phi_ft_579.csv


In [ ]:
import inspect, retrieval as Rtv

# 1) retriever 의 map 들이 제대로 채워졌는지 (패치 후 재생성된 게 맞는지)
print('chunk_text_map:', len(retriever.chunk_text_map))
print('chunk_meta_map:', len(retriever.chunk_meta_map))
k = list(retriever.chunk_text_map.keys())[:2]
print('map 키 샘플:', k)
print('chroma id 가 map 에 있나:', '0001_P_0000_C_0000' in retriever.chunk_text_map)

# 2) BidMateRetriever.__init__ 에서 map 을 어떻게 만드는지
print('\n=== __init__ 소스 ===')
print(inspect.getsource(Rtv.BidMateRetriever.__init__))

chunk_text_map: 1
chunk_meta_map: 1
map 키 샘플: ['']
chroma id 가 map 에 있나: False

=== __init__ 소스 ===
    def __init__(self, collection, bm25_index, bm25_chunk_ids,
                 bm25_texts, embed_model, all_chunks, reranker=None):
        self.collection      = collection
        self.bm25_index      = bm25_index
        self.bm25_chunk_ids  = bm25_chunk_ids
        self.bm25_texts      = bm25_texts
        self.embed_model     = embed_model
        self.chunk_meta_map  = {c["chunk_id"]: c["metadata"] for c in all_chunks}
        self.chunk_text_map  = {c["chunk_id"]: c["text"]     for c in all_chunks}
        self._emb_cache: dict = {}
        self.reranker        = reranker



In [ ]:
# retriever 내부 map 재생성 (패치된 _normalize_chunk 기준) — 재로딩 없이 교체
import retrieval as Rtv

# 패치가 적용됐는지 먼저 확인
import inspect
assert 'child_id' in inspect.getsource(Rtv._normalize_chunk), '❌ _normalize_chunk 패치 안 됨 — 패치 셀 먼저 실행'

# 올바른 id 로 all_chunks 재생성
all_chunks = Rtv.load_chunks()
_ids = [c['chunk_id'] for c in all_chunks[:3]]
assert _ids[0] == '0001_P_0000_C_0000', f'❌ chunk_id 여전히 이상: {_ids}'

# retriever 의 map 두 개 교체
retriever.chunk_meta_map = {c["chunk_id"]: c["metadata"] for c in all_chunks}
retriever.chunk_text_map = {c["chunk_id"]: c["text"]     for c in all_chunks}
Rtv.ALL_AGENCIES = list({c['metadata'].get('agency','') for c in all_chunks if c['metadata'].get('agency','')})

# 검증
print('chunk_text_map:', len(retriever.chunk_text_map))
print('chroma id 가 map 에 있나:', '0001_P_0000_C_0000' in retriever.chunk_text_map)
assert len(retriever.chunk_text_map) > 30000, '❌ map 여전히 비어있음'
print('✅ map 재생성 완료')

chunk_text_map: 38287
chroma id 가 map 에 있나: True
✅ map 재생성 완료


In [ ]:
# [4] 서빙 모듈 로드 + retriever 조립  (← 기존 [4-patch]/[4-patch2]/[4-patch3] 흡수)
import importlib.util, sys, pickle, gc, os, types, math
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import config as C

CODE = '/content/drive/MyDrive/data/bidmate/code'
def load_module(name):
    path = f'{CODE}/{name}.py'
    assert os.path.exists(path), f'파일 없음: {path}'
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
    spec.loader.exec_module(mod); return mod

Rtv = load_module('retrieval')
Rtv.DEVICE = DEVICE
all_chunks = Rtv.load_chunks()
Rtv.ALL_AGENCIES = list({c['metadata'].get(C.AGENCY_KEY,'') for c in all_chunks
                         if c['metadata'].get(C.AGENCY_KEY,'')})
print(f'load_chunks: {len(all_chunks):,} | agencies({C.AGENCY_KEY}): {len(Rtv.ALL_AGENCIES)}')

embed_model = SentenceTransformer(C.EMBED_MODEL_ID, device=DEVICE, cache_folder='/content/hf_cache/hub')
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
collection = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME)
_cnt = collection.count(); print('chroma count:', f'{_cnt:,}')
assert _cnt > 0 and _cnt == C.EXPECT_N, f'❌ chroma count={_cnt:,} (기대 {C.EXPECT_N:,}) — [1] 재실행'

with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
reranker = CrossEncoder(C.RERANKER_ID, device=DEVICE)
retriever = Rtv.BidMateRetriever(
    collection=collection, bm25_index=bm['index'],
    bm25_chunk_ids=bm['chunk_ids'], bm25_texts=bm['texts'],
    embed_model=embed_model, all_chunks=all_chunks, reranker=reranker,
)
Rtv.retriever = retriever

# ── (흡수1) _build_chroma_where: 키매핑 제거, 감지된 기관키 그대로 사용 ──
def _bcw(self, meta_filter):
    if not meta_filter: return None
    conds = []
    for key, val in meta_filter.items():
        if not val: continue
        if isinstance(val, dict):   conds.append({key: val})
        elif isinstance(val, list): conds.append({key: {"$in": [str(v) for v in val]}})
        else:                       conds.append({key: {"$eq": str(val)}})
    if not conds: return None
    return conds[0] if len(conds)==1 else {"$and": conds}
retriever._build_chroma_where = types.MethodType(_bcw, retriever)

# ── (흡수2) retrieve: D타입 sigmoid 컷오프 (임계값은 config.SIG_TH) ──
def _retrieve_sigcut(self, query, meta_filter=None, verbose=False):
    if meta_filter is None:
        meta_filter = __import__('retrieval').parse_metadata_filter(query)
    where           = self._build_chroma_where(meta_filter)
    allowed_indices = self._filter_bm25_ids(meta_filter)
    sub_queries     = self._decompose_query(query)
    if len(sub_queries) > 1:
        dense_ids, sparse_ids = self._multi_retrieve(sub_queries, where, allowed_indices, original_query=query)
    else:
        dense_ids  = self._dense_search(query, where)
        sparse_ids = self._sparse_search(query, allowed_indices)
    ranked  = self._rrf_fusion(dense_ids, sparse_ids)
    boosted = self._soft_boost(ranked)
    boosted = self._mmr_rerank(boosted, query=query)
    boosted = self._rerank(boosted, query=query)

    if len(sub_queries) > 1:
        per_agency = max(2, 5 // len(sub_queries))
        agency_counts, top5 = {}, []
        for cid, score in boosted:
            meta = self.chunk_meta_map.get(cid, {})
            ag = meta.get(C.AGENCY_KEY, meta.get("organization_cleaned", ""))
            if agency_counts.get(ag, 0) < per_agency:
                top5.append((cid, score)); agency_counts[ag] = agency_counts.get(ag, 0) + 1
            if len(top5) >= 5: break
    else:
        top5 = boosted[:5]

    def _sig(x): return 1/(1+math.exp(-x))
    if top5 and _sig(top5[0][1]) < C.SIG_TH:
        top5 = []   # 근거 부족 → D타입 거절

#     return {
#         "context"    : self._build_context(top5),
#         "top_chunks" : [{"rank": i+1, "chunk_id": cid, "boosted_score": sc,
#                          "text": self.chunk_text_map.get(cid,""), "metadata": self.chunk_meta_map.get(cid,{})}
#                         for i, (cid, sc) in enumerate(top5)],
#         "meta_filter": meta_filter, "dense_ids": dense_ids,
#         "sparse_ids" : sparse_ids, "sub_queries": sub_queries,
#     }
# retriever.retrieve = types.MethodType(_retrieve_sigcut, retriever)


    return {
        "context"    : self._build_context(top5),
        "top_chunks" : [{"rank": i+1, "chunk_id": cid, "boosted_score": sc,
                         "text": self.chunk_text_map.get(cid, self.chunk_text_map.get(int(cid) if str(cid).isdigit() else str(cid), "")),
                         "metadata": self.chunk_meta_map.get(cid, self.chunk_meta_map.get(int(cid) if str(cid).isdigit() else str(cid), {}))}
                        for i, (cid, sc) in enumerate(top5)],
        "meta_filter": meta_filter, "dense_ids": dense_ids,
        "sparse_ids" : sparse_ids, "sub_queries": sub_queries,
    }
retriever.retrieve = types.MethodType(_retrieve_sigcut, retriever)


# ── (흡수3) _get_hwp_context: original_name 없으면 source_file 로 alias ──
_orig_get_hwp = Rtv._get_hwp_context
def _get_hwp_src(query, top_chunks, embed_model, top_docs=2):
    for c in top_chunks:
        m = c.get("metadata", {})
        if "original_name" not in m and m.get("source_file"):
            m["original_name"] = m["source_file"]
    return _orig_get_hwp(query, top_chunks, embed_model, top_docs)
Rtv._get_hwp_context = _get_hwp_src

print('✅ retriever 초기화 + 패치 3종 적용 (where/sigcut/hwp-alias)')

# ── (흡수4) _normalize_chunk 패치: kh_v3 는 chunk_id 키가 없고 child_id 사용 ──
import types as _types
def _normalize_chunk_khv3(c: dict) -> dict:
    meta = dict(c.get("metadata", {}))
    if "agency" not in meta:
        meta["agency"] = meta.get("organization_cleaned",
                         meta.get("organization_raw", "미지정"))
    if "source_file" not in meta:
        meta["source_file"] = meta.get("original_name", "")
    if "year" in meta:
        meta["year"] = str(meta["year"])
    meta.setdefault("has_table",  False)
    meta.setdefault("has_number", False)
    chunk_text = c.get("chunk_text", c.get("text", ""))
    summary    = c.get("text", "")
    if summary and summary != chunk_text and "사업예산" in summary:
        full_text = summary + "\n" + chunk_text
    else:
        full_text = chunk_text
    # ★ kh_v3: chunk_id 가 없으면 child_id → parent_id 순으로 사용
    cid = c.get("chunk_id") or c.get("child_id") or c.get("parent_id") or ""
    return {"chunk_id": cid, "text": full_text, "metadata": meta}

Rtv._normalize_chunk = _normalize_chunk_khv3
print('✅ _normalize_chunk 패치 (chunk_id ← child_id)')

load_chunks: 38,287 | agencies(agency): 404


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

chroma count: 38,287


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ retriever 초기화 + 패치 3종 적용 (where/sigcut/hwp-alias)
✅ _normalize_chunk 패치 (chunk_id ← child_id)


In [ ]:
all_chunks = Rtv.load_chunks()
# 검증: chunk_id 가 제대로 채워졌는지
_ids = [c['chunk_id'] for c in all_chunks[:3]]
print('chunk_id 샘플:', _ids)
assert all(_ids) and _ids[0] != '', '❌ chunk_id 여전히 빈값 — child_id 매핑 실패'
assert len(set(c['chunk_id'] for c in all_chunks)) > 30000, '❌ chunk_id 중복 — 고유성 깨짐'
print(f'고유 chunk_id: {len(set(c["chunk_id"] for c in all_chunks)):,}개')

chunk_id 샘플: ['0001_P_0000_C_0000', '0001_P_0000_C_0001', '0001_P_0000_C_0002']
고유 chunk_id: 38,287개


In [ ]:
# [5] generator 로드 (서빙 generation.py + FT Phi LoRA 어댑터)
import torch
Gen = load_module('generation')
generator = Gen.init_generator(Rtv.get_context)
Gen.generator = generator
print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,1),
      '/', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
print('✅ generator 초기화 완료 (Phi-4-mini + LoRA)')

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

VRAM 사용: 31.3 / 42.4 GB
✅ generator 초기화 완료 (Phi-4-mini + LoRA)


In [ ]:
# [6] 스모크 테스트 — 검색이 실제로 문서를 가져오는지 + 메타필터 동작 확인 (0점 조기 감지)
import json, ast
import time, pandas as pd, json, ast
eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('타입 분포:', eval_df['type'].value_counts().to_dict())

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

r = eval_df.iloc[0]
hist = _ph(r.get('history','')); mf = _pm(r.get('metadata_filter',''))

# 1) 검색 단독
rewritten = generator._rewrite_query(r['question'], hist or None)
rr = retriever.retrieve(rewritten, meta_filter=mf)
top = rr.get('top_chunks', [])
print(f'\n[검색] rewritten={rewritten[:50]!r}')
print(f'[검색] top_chunks: {len(top)}')
assert len(top) > 0, '❌ 검색 0건 — chroma 비었거나 메타필터 과도/기관키 불일치. [0b]/[1] 재확인'
names = [c['metadata'].get('source_file','') for c in top]
print(f'[검색] retrieved_names: {names}')

# 2) 메타필터 키 정합 빠른 점검 (필터가 있을 때만)
if mf:
    where = retriever._build_chroma_where(mf)
    got = collection.get(where=where, limit=3)
    print(f'[필터] where={where} → 매칭 {len(got["ids"])}건',
          '' if got['ids'] else '  ⚠️ 0건이면 mf 키/값이 chroma 메타와 불일치')

# 3) 생성 1건
t=time.time()
out = generator.generate(r['question'], history=hist or None, meta_filter=mf)
dt=time.time()-t
print(f'\n[{r["type"]}] {r["question"][:40]}')
print('답변:', out['answer'][:200])
print(f'\n1건 {dt:.1f}초 → 579행 예상 {dt*579/3600:.1f}시간')
print('\n[매칭 점검] GT docs   :', r['ground_truth_docs'])
print('[매칭 점검] retrieved :', json.dumps(names, ensure_ascii=False))

타입 분포: {'B': 214, 'A': 172, 'D': 65, 'E': 65, 'C': 63}

[검색] rewritten="한국가스공사 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 예산 규모입니까?"
[검색] top_chunks: 5
[검색] retrieved_names: ['한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp']
[필터] where={'agency': {'$eq': '한국가스공사'}} → 매칭 3건 

[A] 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 
답변: 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 1,350,000,000원입니다.

[출처]
  [1]  (score: 0.0022)
  [2]  (score: 0.0022)
  [3]  (score: 0.0022)
  [4]  (score: 0.0022)
  [5]  (score: 0.0022)

1건 2.2초 → 579행 예상 0.3시간

[매칭 점검] GT docs   : ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"]
[매칭 점검] retrieved : ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"]


In [ ]:
import os, config as C
p = f'{C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv'
if os.path.exists(p):
    os.remove(p); print('빈 context 생성본 삭제 — [7] 처음부터 재생성')

In [ ]:
# [7] 579행 생성 — question 기준 done 판정, 25행마다 체크포인트
import pandas as pd, json, ast, time, os
from tqdm.auto import tqdm
import config as _C

OUT=_C.OUT_TAG_DIR; os.makedirs(OUT,exist_ok=True)
GEN_PATH=f'{OUT}/e2e_kure_phi_ft_579.csv'

# history/metadata 파서 (이 셀 단독 실행 가능하도록 자체 정의)
def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('평가셋:', len(eval_df), '| 고유 question:', eval_df['question'].nunique(),
      '| 타입:', eval_df['type'].value_counts().to_dict())

done, records = set(), []
if os.path.exists(GEN_PATH):
    prev = pd.read_csv(GEN_PATH)
    prev = prev[prev['answer'].notna() & (prev['answer'].astype(str).str.len()>0)].drop_duplicates(subset='question')
    records = prev.to_dict('records'); done = set(prev['question'])
    print('체크포인트 재사용:', len(done))

pending = eval_df.drop_duplicates(subset='question')
pending = pending[~pending['question'].isin(done)]
print('신규:', len(pending))

for _, row in tqdm(pending.iterrows(), total=len(pending), desc='생성'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    t0=time.time(); rewritten = generator._rewrite_query(q, hist or None)
    rr = retriever.retrieve(rewritten, meta_filter=mf); retr_ms = round((time.time()-t0)*1000)
    top = rr['top_chunks']
    t1=time.time(); out = generator.generate(q, history=hist or None, meta_filter=mf); gen_ms = round((time.time()-t1)*1000)
    records.append({
        'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'question':q,'rewritten_query':rewritten,
        'ground_truth_answer':row['ground_truth_answer'],'ground_truth_docs':row['ground_truth_docs'],
        'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('source_file','') for c in top], ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms,
    })
    if len(records)%25==0:
        pd.DataFrame(records).to_csv(GEN_PATH,index=False,encoding='utf-8-sig')

gen_df=pd.DataFrame(records).drop_duplicates(subset='question')
gen_df.to_csv(GEN_PATH,index=False,encoding='utf-8-sig')
print('✅ 생성 완료:', len(gen_df), '| 고유 question:', gen_df['question'].nunique())

평가셋: 579 | 고유 question: 578 | 타입: {'B': 214, 'A': 172, 'D': 65, 'E': 65, 'C': 63}
신규: 578


생성:   0%|          | 0/578 [00:00<?, ?it/s]

✅ 생성 완료: 578 | 고유 question: 578


In [ ]:
import chromadb
client = chromadb.PersistentClient(path='/content/bidmate_kh_v3/chroma_db_kh_v3_clean')
col = client.get_collection('bidmate_kh_v3_KURE_PHI')
print(col.get(limit=1, include=['metadatas'])['metadatas'])

[{'has_budget': False, 'source_file': '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp', 'year': '2024', 'has_ratio': False, 'has_period': True, 'project_name': '2024년 벤처확인종합관리시스템 기능 고도화 용역사업', 'agency_raw': '(사)벤처기업협회', 'has_number': True, 'section_type': 'table_area', 'has_quantity': False, 'domains': '벤처확인종합관리시스템,업무시스템', 'agency': '벤처기업협회', 'agency_cleaned': '벤처기업협회', 'has_table': True}]


In [ ]:
import pandas as pd, json, config as C
df = pd.read_csv(f'{C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv')

r = df.iloc[0]
print('=== 첫 행 ===')
print('answer[:150]:', str(r['answer'])[:150])
print('retrieved_names:', repr(r['retrieved_names']))
print('retrieved_context[:200]:', repr(str(r['retrieved_context'])[:200]))
print('retrieved_scores:', repr(r['retrieved_scores']))
print()
# context 가 비었는지 vs names 만 비었는지
empty_ctx = df['retrieved_context'].astype(str).str.strip().isin(['','nan']).sum()
print(f'retrieved_context 빈 행: {empty_ctx} / {len(df)}')
# 답변이 "찾을 수 없습니다" 류인지
reject = df['answer'].astype(str).str.contains('찾을 수 없|확인할 수 없', na=False).sum()
print(f'거절성 답변 행: {reject} / {len(df)}')

=== 첫 행 ===
answer[:150]: 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 1,000,000,000원입니다.

[출처]
  [1]  (score: 0.0022)
  [2]  (score: 0.0022)
  [3]  (score: 0.0022)
  [4]  (score: 0
retrieved_names: '["", "", "", "", ""]'
retrieved_context[:200]: '[1] \n(출처: 미상  | score: 0.0022)\n\n[2] \n(출처: 미상  | score: 0.0022)\n\n[3] \n(출처: 미상  | score: 0.0022)\n\n[4] \n(출처: 미상  | score: 0.0022)\n\n[5] \n(출처: 미상  | score: 0.0022)'
retrieved_scores: '[0.002213751897215843, 0.002213751897215843, 0.002213751897215843, 0.002213751897215843, 0.002213751897215843]'

retrieved_context 빈 행: 1 / 578
거절성 답변 행: 4 / 578


In [ ]:
import config as C
# 1) chroma 가 반환하는 실제 id 형태
sample = collection.get(limit=3, include=['metadatas','documents'])
print('=== chroma 내부 id (3개) ===')
for i, _id in enumerate(sample['ids']):
    meta = sample['metadatas'][i] if sample['metadatas'] else {}
    doc  = sample['documents'][i] if sample['documents'] else None
    print(f'  id={_id!r}')
    print(f'    메타키={list(meta.keys())[:8]}')
    print(f'    source_file={meta.get("source_file")!r} | doc유무={bool(doc)} doc[:50]={str(doc)[:50]!r}')

# 2) retriever 의 map 키 형태
print('\n=== retriever chunk_text_map 키 (3개) ===')
keys = list(retriever.chunk_text_map.keys())[:3]
for k in keys:
    print(f'  key={k!r} | text[:40]={retriever.chunk_text_map[k][:40]!r}')
print(f'\nmap 크기: text={len(retriever.chunk_text_map):,} meta={len(retriever.chunk_meta_map):,}')

# 3) chroma id 가 map 에 있는지 직접 대조
chroma_ids = set(sample['ids'])
map_keys = set(retriever.chunk_text_map.keys())
print('chroma id ∈ map?:', chroma_ids & map_keys, '(교집합 비어있으면 id 체계 불일치 확정)')

=== chroma 내부 id (3개) ===
  id='0001_P_0000_C_0000'
    메타키=['has_ratio', 'has_number', 'source_file', 'has_period', 'agency', 'has_table', 'year', 'has_budget']
    source_file='(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp' | doc유무=True doc[:50]='[TABLE]\n\n2024. 03.\n\n[IMAGE]\n\n[TABLE]\n\n[TABLE]\n\n[TA'
  id='0001_P_0000_C_0001'
    메타키=['has_number', 'has_ratio', 'section_type', 'has_table', 'year', 'has_budget', 'has_quantity', 'project_name']
    source_file='(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp' | doc유무=True doc[:50]='권리를 부여하는 제도\n\n-중소벤처24 內 스톡옵션 신청 기능 이관을 추진하며, 과거 데이터'
  id='0001_P_0000_C_0002'
    메타키=['domains', 'has_period', 'has_quantity', 'has_budget', 'agency', 'agency_cleaned', 'project_name', 'year']
    source_file='(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp' | doc유무=True doc[:50]='2,000,000원(부가가치세 포함)\n\n□ (계약방식) 제한경쟁입찰(협상에 의한 계약)에 '

=== retriever chunk_text_map 키 (3개) ===
  key='' | text[:40]='특별 보안교육\n실시\n경미\n1. 업무 관련 서류 관리 소홀\n가. 진행 중인'

map 크기: text=

In [ ]:
import json, config as C
# 1) kh_v3.json 실제 구조
with open(C.CHUNKS_PATH, encoding='utf-8') as f:
    raw = json.load(f)
print('타입:', type(raw).__name__)
if isinstance(raw, list):
    print('길이:', len(raw))
    print('첫 원소 타입:', type(raw[0]).__name__)
    print('첫 원소 키:', list(raw[0].keys())[:12] if isinstance(raw[0], dict) else raw[0][:80])
elif isinstance(raw, dict):
    print('최상위 키:', list(raw.keys())[:10])
    fk = list(raw.keys())[0]
    print(f'첫 값({fk}) 타입:', type(raw[fk]).__name__)
    if isinstance(raw[fk], dict): print('  키:', list(raw[fk].keys())[:12])

# 2) load_chunks() 가 뭘 반환했는지
print('\nall_chunks 길이:', len(all_chunks))
if all_chunks:
    print('첫 청크 타입:', type(all_chunks[0]).__name__)
    if isinstance(all_chunks[0], dict):
        print('첫 청크 키:', list(all_chunks[0].keys()))
        print('chunk_id 값:', all_chunks[0].get('chunk_id', all_chunks[0].get('id', '없음')))

# 3) load_chunks 소스 확인 — id/text 를 어느 키에서 뽑는지
import inspect, retrieval as Rtv
print('\n=== load_chunks 소스 ===')
print(inspect.getsource(Rtv.load_chunks))

타입: list
길이: 38287
첫 원소 타입: dict
첫 원소 키: ['parent_id', 'child_id', 'doc_id', 'text', 'parent_text', 'metadata']

all_chunks 길이: 38287
첫 청크 타입: dict
첫 청크 키: ['chunk_id', 'text', 'metadata']
chunk_id 값: 

=== load_chunks 소스 ===
def load_chunks() -> list:
    path = Path(CHUNKS_PATH)
    if not path.exists():
        raise FileNotFoundError(f"청크 파일 없음: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    raw = data if isinstance(data, list) else [data]
    return [_normalize_chunk(c) for c in raw]



In [ ]:
import inspect, retrieval as Rtv
print('=== _normalize_chunk 소스 ===')
print(inspect.getsource(Rtv._normalize_chunk))

# kh_v3.json 첫 청크의 id 관련 필드 실제 값
import json, config as C
with open(C.CHUNKS_PATH, encoding='utf-8') as f:
    raw = json.load(f)
c0 = raw[0]
print('\n=== kh_v3.json 첫 청크 id 필드 ===')
for k in ['parent_id','child_id','doc_id']:
    print(f'  {k} = {c0.get(k)!r}')
print('  metadata 키:', list(c0['metadata'].keys())[:12])
print('  metadata 안 id 후보:', {k:v for k,v in c0['metadata'].items() if 'id' in k.lower()})

# chroma id 형식과 비교: '0001_P_0000_C_0000'
# child_id 나 parent_id+child_id 조합이 이 형식인지 확인
print('\n  → chroma id 예시: 0001_P_0000_C_0000')

=== _normalize_chunk 소스 ===
def _normalize_chunk(c: dict) -> dict:
    meta = dict(c.get("metadata", {}))
    if "agency" not in meta:
        meta["agency"] = meta.get("organization_cleaned",
                         meta.get("organization_raw", "미지정"))
    if "source_file" not in meta:
        meta["source_file"] = meta.get("original_name", "")
    if "year" in meta:
        meta["year"] = str(meta["year"])
    meta.setdefault("has_table",  False)
    meta.setdefault("has_number", False)
    chunk_text = c.get("chunk_text", c.get("text", ""))
    summary    = c.get("text", "")  # 예산/기간 요약
    if summary and summary != chunk_text and "사업예산" in summary:
        full_text = summary + "\n" + chunk_text
    else:
        full_text = chunk_text
        full_text = chunk_text
    return {
        "chunk_id": c.get("chunk_id", ""),
        "text"    : full_text,
        "metadata": meta,
    }


=== kh_v3.json 첫 청크 id 필드 ===
  parent_id = '0001_P_0000'
  child_id = '0001_P_0000_C_0000'
  doc

In [ ]:
# [8] 생성 결과 무결성 점검 (question 기준)
import pandas as pd, json
import config as _C
GEN_PATH = f'{_C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv'
df = pd.read_csv(GEN_PATH); ev = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

print('저장된 행:', len(df), '| 고유 question:', df['question'].nunique())
empty_ans = df['answer'].isna().sum() + (df['answer'].astype(str).str.len()==0).sum()
print('answer 빈 행:', empty_ans)
print('eval 에 있는데 저장 안 된 question:', len(set(ev['question']) - set(df['question'])))
print('중복 question:', df['question'].duplicated().sum())

def _empty_names(raw):
    try: v=json.loads(raw)
    except Exception: return True
    return (not isinstance(v,list)) or len(v)==0 or all(not str(x).strip() for x in v)
n_empty = df['retrieved_names'].apply(_empty_names).sum()
print(f'retrieved_names 비어있는 행: {n_empty} / {len(df)}')
assert n_empty < len(df)*0.5, '❌ 검색결과가 절반 이상 비어있음 — chroma/메타필터 재점검'
print('✅ 무결성 통과')

저장된 행: 578 | 고유 question: 578
answer 빈 행: 0
eval 에 있는데 저장 안 된 question: 0
중복 question: 0
retrieved_names 비어있는 행: 1 / 578
✅ 무결성 통과


In [ ]:
# [9] Retrieval 지표 — Hit@5 / MRR / nDCG
import pandas as pd, json, ast, math, os
import config as _C
OUT=_C.OUT_TAG_DIR
gen_df = pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv')

def _tolist(raw):
    if isinstance(raw,list): return raw
    for fn in (json.loads, ast.literal_eval):
        try:
            v=fn(raw)
            if isinstance(v,list): return v
        except Exception: pass
    return []
def _norm(x): return os.path.splitext(str(x).strip())[0].replace(' ','').lower()

def rmetrics(row, k=5):
    gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
    got=[_norm(x) for x in _tolist(row['retrieved_names'])][:k]
    if not gts: return None
    rank=next((i for i,g in enumerate(got,1) if g in gts), 0)
    dcg=sum(1.0/math.log2(i+1) for i,g in enumerate(got,1) if g in gts)
    idcg=sum(1.0/math.log2(i+1) for i in range(1,min(len(gts),k)+1))
    return pd.Series({'hit@5':1.0 if rank else 0.0,'mrr':1.0/rank if rank else 0.0,'ndcg':dcg/idcg if idcg else 0.0})

rm = gen_df.join(gen_df.apply(rmetrics, axis=1))
valid = rm.dropna(subset=['hit@5'])
print(f'대상 {len(valid)}행')
print('전체:', valid[['hit@5','mrr','ndcg']].mean().round(4).to_dict())
print('\n타입별:\n', valid.groupby('type')[['hit@5','mrr','ndcg']].mean().round(4))

if valid['hit@5'].mean() == 0.0:
    print('\n⚠️ Hit@5 전체 0 — 매칭 진단(정규화 후):')
    for _, row in valid.head(3).iterrows():
        gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
        got=[_norm(x) for x in _tolist(row['retrieved_names'])][:5]
        print('  GT :', gts); print('  GOT:', got); print('  교집합:', set(gts)&set(got), '\n')
    print('  → 형태 다르면 _norm() 규칙(확장자/공백/구분자) 조정')

s = valid.groupby('type')[['hit@5','mrr','ndcg']].mean()
s.loc['ALL'] = valid[['hit@5','mrr','ndcg']].mean()
s.to_csv(f'{OUT}/retrieval_metrics_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

대상 573행
전체: {'hit@5': 0.9005, 'mrr': 0.8147, 'ndcg': 1.6009}

타입별:
        hit@5     mrr    ndcg
type                        
A     0.9415  0.8647  2.0675
B     0.9623  0.8469  1.1215
C     0.8387  0.7742  1.8299
D     0.8750  0.7982  1.8688
E     0.6719  0.6302  1.4523
✅ 저장


In [ ]:
# [10] Generation Judge (gpt-5.4-mini async, 6지표, 50행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
from openai import AsyncOpenAI
from tqdm.auto import tqdm
import config as _C
nest_asyncio.apply()
OUT=_C.OUT_TAG_DIR

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'),'OPENAI_API_KEY 필요'
_M='gpt-5.4-mini'; _client=AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM=asyncio.Semaphore(15); _RETRY=3

_JP={
'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n"
                "5점: 모든 내용이 Context 근거. 1점: Context 무관/날조.\n[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n"
             "5점: 핵심을 정확·간결히 해결. 1점: 동문서답.\n[Question]\n{query}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n"
             "5점: 근거 없으면 적절히 거절. 1점: 근거 없이 날조.\n[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n"
               "5점: 모두 일치. 1점: 핵심 불일치.\n[Ground Truth]\n{ground_truth}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n"
                     "5점: 모두 필요. 1점: 대부분 불필요.\n[Question]\n{query}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n"
                  "5점: 모든 핵심 포함. 1점: 누락 심각.\n[Ground Truth]\n{ground_truth}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
}

def _parse(raw):
    if not raw: return None
    m=re.search(r'점수\s*:\s*(\d)',raw)
    if m: return int(m.group(1))
    s=raw.strip()
    if s.isdigit() and 1<=int(s)<=5: return int(s)
    d=re.findall(r'\b[1-5]\b',raw); return int(d[0]) if d else None

async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r=await _client.chat.completions.create(model=_M,
                    messages=[{'role':'user','content':prompt}], max_completion_tokens=20, timeout=15)
                return _parse(r.choices[0].message.content)
            except Exception:
                if a==_RETRY-1: return None
                await asyncio.sleep(2**a)

async def score_one(q,ctx,ans,gt=None):
    tasks,none_keys={},[]
    for m in ('faithfulness','relevance','rejection'):
        tasks[m]=_ask(_JP[m].format(context=ctx,query=q,answer=ans))
    for m in ('correctness','context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m]=_ask(_JP[m].format(ground_truth=gt,answer=ans,context=ctx))
        else: none_keys.append(m)
    tasks['context_precision']=_ask(_JP['context_precision'].format(query=q,context=ctx))
    vals=await asyncio.gather(*tasks.values())
    res=dict(zip(tasks.keys(),vals))
    for k in none_keys: res[k]=None
    return res

_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
JUDGE_PATH=f'{OUT}/quant_scores_kure_phi_ft.csv'

async def run_judge():
    gdf=pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv').drop_duplicates(subset='question')
    done,rows=set(),[]
    if os.path.exists(JUDGE_PATH):
        ck=pd.read_csv(JUDGE_PATH); ck=ck[ck['relevance'].notna()].drop_duplicates(subset='question')
        rows=ck.to_dict('records'); done=set(ck['question']); print('judge 체크포인트:',len(done))
    pending=gdf[~gdf['question'].isin(done)]; print('judge 신규:',len(pending))
    for _,row in tqdm(pending.iterrows(), total=len(pending), desc='judge'):
        ans=row['answer']
        base={'id':row['id'],'question':row['question'],'type':row['type'],'difficulty':row['difficulty']}
        if not isinstance(ans,str) or '오류' in str(ans)[:30]:
            for m in _MET: base[m]=None
        else:
            base.update(await score_one(row['question'],row['retrieved_context'],ans,row.get('ground_truth_answer')))
        rows.append(base)
        if len(rows)%50==0: pd.DataFrame(rows).to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out=pd.DataFrame(rows).drop_duplicates(subset='question')
    out.to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    print('✅ judge 완료:',len(out)); return out

judge_df = asyncio.get_event_loop().run_until_complete(run_judge())

judge 신규: 578


judge:   0%|          | 0/578 [00:00<?, ?it/s]

✅ judge 완료: 578


In [ ]:
# [11] Generation 요약
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
judge_df=pd.read_csv(f'{OUT}/quant_scores_kure_phi_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s=judge_df.groupby('type')[_MET].mean(); s.loc['ALL']=judge_df[_MET].mean()
s.round(3).to_csv(f'{OUT}/generation_summary_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

전체: {'faithfulness': 1.749, 'relevance': 2.139, 'rejection': 1.582, 'correctness': 2.314, 'context_precision': 3.293, 'context_recall': 3.049}

타입별:
       faithfulness  relevance  rejection  correctness  context_precision  \
type                                                                       
A            1.872      2.529      1.628        2.145              3.797   
B            1.690      1.967      1.385        2.873              3.075   
C            1.823      2.145      1.629        2.016              3.677   
D            1.138      2.231      1.692        1.523              1.862   
E            2.154      1.569      1.954        2.000              3.738   

      context_recall  
type                  
A              3.343  
B              2.676  
C              3.597  
D              2.708  
E              3.308  
✅ 저장


In [ ]:
# [12] Release Gate
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
retr=pd.read_csv(f'{OUT}/retrieval_metrics_kure_phi_ft.csv',index_col=0)
genm=pd.read_csv(f'{OUT}/generation_summary_kure_phi_ft.csv',index_col=0)
def v(x,p,g): return 'GOOD' if x>=g else ('PASS' if x>=p else 'FAIL')

print('RETRIEVAL (전체)')
print(f"  Hit@5 {retr.loc['ALL','hit@5']:.3f} → {v(retr.loc['ALL','hit@5'],0.90,0.95)}")
print(f"  MRR   {retr.loc['ALL','mrr']:.3f} → {v(retr.loc['ALL','mrr'],0.82,0.87)}")
print(f"  nDCG  {retr.loc['ALL','ndcg']:.3f} → {v(retr.loc['ALL','ndcg'],0.78,0.83)}")
print('타입별 MRR')
for t,(p,g) in {'A':(0.92,0.95),'B':(0.77,0.82),'C':(0.88,0.93),'D':(0.81,0.86),'E':(0.82,0.87)}.items():
    if t in retr.index: print(f"  {t} {retr.loc[t,'mrr']:.3f} → {v(retr.loc[t,'mrr'],p,g)}")
print('GENERATION (≥3.5 PASS / ≥4.0 GOOD)')
for m in ['faithfulness','relevance','rejection','context_precision']:
    if m in genm.columns:
        print(f"  {m:18} {genm.loc['ALL',m]:.3f} → {v(genm.loc['ALL',m],3.5,4.0)}")

RETRIEVAL (전체)
  Hit@5 0.901 → PASS
  MRR   0.815 → FAIL
  nDCG  1.601 → GOOD
타입별 MRR
  A 0.865 → FAIL
  B 0.847 → GOOD
  C 0.774 → FAIL
  D 0.798 → FAIL
  E 0.630 → FAIL
GENERATION (≥3.5 PASS / ≥4.0 GOOD)
  faithfulness       1.749 → FAIL
  relevance          2.139 → FAIL
  rejection          1.582 → FAIL
  context_precision  3.293 → FAIL


In [ ]:
# [13] 산출물 목록
import os
import config as _C
OUT=_C.OUT_TAG_DIR
for f in sorted(os.listdir(OUT)):
    p=os.path.join(OUT,f)
    print(f'{os.path.getsize(p)/1024:8.1f} KB  {f}' if os.path.isfile(p) else f'{"<dir>":>11}  {f}')

  4579.5 KB  e2e_kure_phi_ft_579.csv
     0.3 KB  generation_summary_kure_phi_ft.csv
   176.3 KB  quant_scores_kure_phi_ft.csv
     0.3 KB  retrieval_metrics_kure_phi_ft.csv


In [ ]:
# [14] 정성 분석 — 오류 역추적 / C타입 맥락 / 타입별 요약
import pandas as pd, os
import config as _C
OUT=_C.OUT_TAG_DIR; QUAL_DIR=f'{OUT}/qual'; os.makedirs(QUAL_DIR, exist_ok=True)
gen_df   = pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv').drop_duplicates(subset='question')
judge_df = pd.read_csv(f'{OUT}/quant_scores_kure_phi_ft.csv').drop_duplicates(subset='question')
_MET = ['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
ERROR_TH = 3.0

score_cols = ['question'] + [m for m in _MET if m in judge_df.columns]
merged = gen_df.merge(judge_df[score_cols], on='question', how='left')
print(f'병합: {len(merged)}행')

# 1) 오류 역추적
mask = pd.Series(False, index=merged.index)
for m in ['faithfulness','relevance']:
    if m in merged.columns: mask |= (merged[m].notna() & (merged[m] <= ERROR_TH))
mask |= merged['answer'].astype(str).str.contains('오류', na=False)
err_cols = [c for c in ['id','type','difficulty','question','ground_truth_answer','answer','retrieved_context']+_MET if c in merged.columns]
merged[mask][err_cols].to_csv(f'{QUAL_DIR}/qual_error_analysis.csv', index=False, encoding='utf-8-sig')
print(f'1) 오류 케이스: {int(mask.sum())}건 → qual_error_analysis.csv')

# 2) C타입 맥락 추적
kws = ['그 ','저 ','위에서','앞서','아까','해당','그것','거기']
cmask = (merged['type']=='C') | merged['question'].astype(str).str.contains('|'.join(kws), na=False, regex=True)
c_cols = [c for c in ['id','type','question','ground_truth_answer','answer','retrieved_context'] if c in merged.columns]
merged[cmask][c_cols].to_csv(f'{QUAL_DIR}/qual_ctype_tracking.csv', index=False, encoding='utf-8-sig')
print(f'2) C타입/맥락: {int(cmask.sum())}건 → qual_ctype_tracking.csv')

# 3) 타입별 요약
rows=[]
for t in ['A','B','C','D','E']:
    sub = merged[merged['type']==t]
    if sub.empty: continue
    r={'type':t,'n':len(sub)}
    for m in _MET: r[m]=round(sub[m].dropna().mean(),3) if m in sub.columns else None
    r['gen_errors']=int(sub['answer'].astype(str).str.contains('오류', na=False).sum())
    rows.append(r)
summary_df=pd.DataFrame(rows)
summary_df.to_csv(f'{QUAL_DIR}/qual_summary.csv', index=False, encoding='utf-8-sig')
print('3) 타입별 요약 → qual_summary.csv\n')
print(summary_df.to_string(index=False))
print(f'\n✅ 정성 분석 저장: {QUAL_DIR}/')

병합: 578행
1) 오류 케이스: 563건 → qual_error_analysis.csv
2) C타입/맥락: 120건 → qual_ctype_tracking.csv
3) 타입별 요약 → qual_summary.csv

type   n  faithfulness  relevance  rejection  correctness  context_precision  context_recall  gen_errors
   A 172         1.872      2.529      1.628        2.145              3.797           3.343           2
   B 213         1.690      1.967      1.385        2.873              3.075           2.676           2
   C  63         1.823      2.145      1.629        2.016              3.677           3.597           2
   D  65         1.138      2.231      1.692        1.523              1.862           2.708           1
   E  65         2.154      1.569      1.954        2.000              3.738           3.308           0

✅ 정성 분석 저장: /content/bidmate/outputs/kh_v3/qual/


In [ ]:
# chroma 복구본을 tar 로 다시 묶어 Drive 에 저장 (다음 실행부터 정상 데이터 사용)
import os, tarfile, shutil, gc, chromadb, config as C

root = str(C.CHROMA_PATH)                       # /content/bidmate_kh_v3/chroma_db_kh_v3_clean
parent = os.path.dirname(root)                  # /content/bidmate_kh_v3
base = os.path.basename(root)                   # chroma_db_kh_v3_clean

# 0) chroma 핸들 정리 (파일 잠금 해제)
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass

# 1) 묶기 전 정리 — 껍데기 백업 폴더 + ._ 잔재 제거
bak = os.path.join(root, '21be2de2-405f-46c0-b122-611962dac9bb__shell_backup')
if os.path.isdir(bak):
    shutil.rmtree(bak); print('껍데기 백업 폴더 제거')
rm = 0
for dp, _, fs in os.walk(root):
    for f in fs:
        if f.startswith('._'):
            os.remove(os.path.join(dp, f)); rm += 1
print(f'._ 잔재 제거: {rm}개')

# 2) 정상 작동 재확인 (묶기 전 마지막 검증)
col = chromadb.PersistentClient(path=root).get_collection(C.COLLECTION_NAME)
assert col.count() == C.EXPECT_N, f'count 불일치 {col.count()} != {C.EXPECT_N}'
print(f'검증 OK: {C.COLLECTION_NAME} count={col.count():,}')
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass

# 3) tar 생성 (parent 기준으로 묶어야 tar 안 경로가 'chroma_db_kh_v3_clean/...' 가 됨)
LOCAL_TAR = '/content/kh_v3_chroma_FIXED.tar.gz'
print('tar 생성 중... (5GB sqlite 포함이라 수 분 소요)')
with tarfile.open(LOCAL_TAR, 'w:gz') as t:
    t.add(root, arcname=base)
sz = os.path.getsize(LOCAL_TAR) / 1e6
print(f'tar 생성 완료: {LOCAL_TAR} ({sz:.0f}MB)')

# 4) 무결성 검증 — 끝까지 읽혀야 정상 (깨진 tar 재발 방지)
with tarfile.open(LOCAL_TAR) as t:
    names = t.getnames()
assert any('chroma.sqlite3' in n for n in names), 'sqlite 누락'
assert any('data_level0.bin' in n for n in names), 'vector bin 누락'
print(f'tar 검증 OK: {len(names)}개 항목')

# 5) Drive 로 복사 (기존 깨진 tar 덮어쓰기)
DST = f'{C.PROJECT_ROOT.parent if False else "/content/drive/MyDrive/data/bidmate"}/kh_v3_chroma_FIXED.tar.gz'
DST = '/content/drive/MyDrive/data/bidmate/kh_v3_chroma_FIXED.tar.gz'
shutil.copy(LOCAL_TAR, DST)
print(f'Drive 저장 완료: {DST} ({os.path.getsize(DST)/1e6:.0f}MB)')
print('\n다음 실행부터는 [1] 셀의 CHROMA_TAR 을 "kh_v3_chroma_FIXED.tar.gz" 로 바꾸세요.')

껍데기 백업 폴더 제거
._ 잔재 제거: 0개
검증 OK: bidmate_kh_v3_KURE_PHI count=38,287
tar 생성 중... (5GB sqlite 포함이라 수 분 소요)
tar 생성 완료: /content/kh_v3_chroma_FIXED.tar.gz (1786MB)
tar 검증 OK: 8개 항목
Drive 저장 완료: /content/drive/MyDrive/data/bidmate/kh_v3_chroma_FIXED.tar.gz (1786MB)

다음 실행부터는 [1] 셀의 CHROMA_TAR 을 "kh_v3_chroma_FIXED.tar.gz" 로 바꾸세요.


In [ ]:
# [15] 산출물 Drive 백업 — outputs/kh_v3 통째로 (로컬은 런타임 종료 시 소실되므로)
import shutil, os
SRC = '/content/bidmate/outputs/kh_v3'
DST = '/content/drive/MyDrive/data/bidmate/outputs/kh_v3'

# Drive 마운트 확인 (안 돼 있으면 마운트)
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

assert os.path.isdir(SRC), f'원본 없음: {SRC} — [7] 생성을 먼저 실행하세요'
shutil.copytree(SRC, DST, dirs_exist_ok=True)
print('Drive 저장 완료:', DST)
!ls -la "{DST}"
# qual 폴더는 [14] 정성분석을 돌려야 생김 — 있을 때만 출력
if os.path.isdir(f'{DST}/qual'):
    !ls -la "{DST}/qual"
else:
    print('(qual 폴더 없음 — [14] 정성분석 미실행)')

Drive 저장 완료: /content/drive/MyDrive/data/bidmate/outputs/kh_v3
total 4762
-rw------- 1 root root 4689430 Jun  2 06:50 e2e_kure_phi_ft_579.csv
-rw------- 1 root root     313 Jun  2 07:08 generation_summary_kure_phi_ft.csv
drwx------ 2 root root    4096 Jun  2 07:08 qual
-rw------- 1 root root  180572 Jun  2 07:08 quant_scores_kure_phi_ft.csv
-rw------- 1 root root     354 Jun  2 06:52 retrieval_metrics_kure_phi_ft.csv
total 4798
-rw------- 1 root root  875197 Jun  2 07:08 qual_ctype_tracking.csv
-rw------- 1 root root 4036947 Jun  2 07:08 qual_error_analysis.csv
-rw------- 1 root root     313 Jun  2 07:08 qual_summary.csv
